In [2]:
# Importar librerías
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)

In [4]:
pbp = pd.read_csv("C:/Users/jorge/OneDrive/Escritorio/Portafolio/UCI_Credit_Card/UCI_Credit_Card.csv")

In [5]:
pbp.head()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month
0,1,20000.0,2,2,1,24,2,2,-1,-1,-2,-2,3913.0,3102.0,689.0,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1
1,2,120000.0,2,2,2,26,-1,2,0,0,0,2,2682.0,1725.0,2682.0,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1
2,3,90000.0,2,2,2,34,0,0,0,0,0,0,29239.0,14027.0,13559.0,14331.0,14948.0,15549.0,1518.0,1500.0,1000.0,1000.0,1000.0,5000.0,0
3,4,50000.0,2,2,1,37,0,0,0,0,0,0,46990.0,48233.0,49291.0,28314.0,28959.0,29547.0,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0,0
4,5,50000.0,1,2,1,57,-1,0,-1,0,0,0,8617.0,5670.0,35835.0,20940.0,19146.0,19131.0,2000.0,36681.0,10000.0,9000.0,689.0,679.0,0


In [6]:
# --- Valores únicos por columna ---
valores_unicos = []
for col in pbp.columns:
    unicos = pbp[col].dropna().unique()
    valores_unicos.append({
        "Columna": col,
        "Cantidad de valores únicos": len(unicos),
        "Valores únicos": list(unicos)
    })
df_unicos = pd.DataFrame(valores_unicos)

# --- Resumen de nulos ---
df_nulos = pbp.isna().sum().reset_index()
df_nulos.columns = ["Columna", "Cantidad de nulos"]

# --- Resumen de atípicos ---
resumen_outliers = []
for col in pbp.select_dtypes(include='number').columns:
    pbp_mean = pbp[col].mean(skipna=True)
    pbp_desv = pbp[col].std(skipna=True)

    if pbp[col].notna().sum() > 1 and pbp_desv != 0:
        z_score = (pbp[col] - pbp_mean) / pbp_desv
        max_abs = z_score.abs().max(skipna=True)

        if pd.notna(max_abs):
            max_abs_int = int(max_abs)
            valores_unicos = pbp.loc[z_score.abs() >= max_abs_int, col].unique()
            resumen_outliers.append({
                "Columna": col,
                "Max Zscore (abs, entero)": max_abs_int,
                "Cantidad de valores únicos": len(valores_unicos),
                "Valores únicos": list(valores_unicos)
            })
df_outliers = pd.DataFrame(resumen_outliers)

# --- Exportar todo a un mismo Excel ---
with pd.ExcelWriter("C:/Users/jorge/OneDrive/Escritorio/Portafolio/UCI_Credit_Card/resumen_limpieza.xlsx") as writer:
    df_unicos.to_excel(writer, sheet_name="Valores Únicos", index=False)
    df_nulos.to_excel(writer, sheet_name="Resumen Nulos", index=False)
    df_outliers.to_excel(writer, sheet_name="Resumen Atípicos", index=False)


In [ ]:
# Supongamos que tienes un DataFrame llamado df con las columnas:
# 'LIMIT_BAL', 'AGE', 'PAY_0', 'BILL_AMT1', 'default_payment_next_month'

# Variables independientes y dependiente
X = pbp[['LIMIT_BAL','AGE','PAY_0','BILL_AMT1']]
y = pbp['default_payment_next_month']

# División entrenamiento/prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Modelo de regresión logística
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train, y_train)

# Predicciones
y_pred = log_reg.predict(X_test)
y_prob = log_reg.predict_proba(X_test)[:,1]

# Evaluación
print("Matriz de confusión:\n", confusion_matrix(y_test, y_pred))
print("\nReporte de clasificación:\n", classification_report(y_test, y_pred, digits=3))

# Curva ROC
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0,1], [0,1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Curva ROC - Default de Tarjeta')
plt.legend(loc="lower right")
plt.show()

# Coeficientes interpretables
coef_df = pd.DataFrame({
    'Variable': X.columns,
    'Coeficiente': log_reg.coef_[0]
})
print("\nCoeficientes del modelo:\n", coef_df)